<a href="https://colab.research.google.com/github/anamitra-tech/ML-Projects/blob/main/AntiDroneGNB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip -q "/content/drive/MyDrive/f4c2b4n755-1.zip" -d "/content/drive/MyDrive/DroneRF"

In [3]:
!pip install rarfile

In [4]:
!find /content/drive/MyDrive -name "*.rar"

/content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L1.rar
/content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_H1.rar
/content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L2.rar
/content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/FR Data_00000_H2.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_L.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_H.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_L.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_H.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10010_L.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10001_L.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10001_H.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10010_H.rar
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10110_L.rar
/content/drive/MyDrive/Drone

In [5]:
import os
from pathlib import Path

base = Path("/content/drive/MyDrive/DroneRF/DroneRF")

for rar_file in base.rglob("*.rar"):
    print(f"Extracting: {rar_file}")
    os.system(f'unrar x -o+ "{rar_file}" "{rar_file.parent}/"')

Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L1.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_H1.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L2.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/FR Data_00000_H2.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_H.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_H.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10010_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10001_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10001_H.rar
Extracting: /content/drive/MyDrive/

In [6]:
# ════════════════════════════════════════════════════════════════════════════
# CONTINUAL RF DRONE MONITORING SYSTEM — v5
# ════════════════════════════════════════════════════════════════════════════
#
# ALL CHANGES FROM v4 BASELINE:
#
# FIX-1  WINDOW SIZE: 1024 → 8192 samples (0.10ms → 0.82ms per segment).
#         At 10 MHz the drone control link runs at ~10 kHz; you need ≥2 full
#         cycles to extract meaningful instantaneous frequency features.
#         1024 samples captured <1 cycle → ifreq features were pure noise.
#
# FIX-2  SKIP SEGMENT-0 ZERO-PAD: every DroneRF file starts with zeros
#         (SDR hardware ramp-up). raw[0:1024] = [0,0,0,...]. The Hilbert
#         transform of a zero-padded ramp creates a sharp transient that
#         looks like a drone signal → Seg-1 was always FRIENDLY_DRONE even
#         on background files. Fix: start extraction at WINDOW_SIZE not 0.
#
# FIX-3  CONFIDENCE GATE: 0.65 → 0.50. With RF F1=63% many genuine drone
#         segments score 0.50–0.64 and were silently routed to BACKGROUND.
#         Lowering the gate to 0.50 lets borderline segments enter the
#         trust/threat path instead of being swallowed by background.
#
# FIX-4  SYNTHETIC PROFILES: z-values → raw feature-space values.
#         Old code set signal_power_db_z=2.5 in ±2 normalised space, but
#         classify_signal() applies RobustScaler → near-zero → confident
#         drone classification → trust/threat path bypassed entirely.
#         Fix: compute per-feature median and IQR from the saved CSV after
#         extraction, then set profiles as median ± z*IQR in raw space.
#
# FIX-5  AUTO-CLASSIFY UNKNOWNS ON PROMOTION: when an emitter accumulates
#         enough observations to be trustworthy, run the trained RF+GNB
#         classifier on its mean feature vector. If confidence ≥ 0.50,
#         auto-label it as the predicted drone type in the fingerprint DB.
#         This eliminates the manual "SAFE_UNKNOWN_001" generic label.
#
# FIX-6  DYNAMIC TRUSTED DATABASE: the DB now stores classifier prediction,
#         confidence, and promotion timestamp. Entries upgrade from
#         SAFE_UNKNOWN → identified class as more observations arrive.
#         The Bayesian fusion uses LogisticRegression instead of GNB as
#         the lightweight ensemble partner (GNB performs at random here).
#
# FIX-7  USE ALL 52 FEATURES (no MI cutoff). MI selects power/amplitude
#         features that separate BG from drone well but miss the spectral
#         shape features (spec_flatness, l_kurtosis, stft_entropy) that
#         separate AR from Phantom. Pass all 52 to the classifier.
#
# ════════════════════════════════════════════════════════════════════════════

import pathlib
for _old in ["dronerf_features_v4.csv", "rf_fingerprint_db.json"]:
    # NOTE: dronerf_features_v5.csv is kept — extraction was successful
    _p = pathlib.Path(_old)
    if _p.exists():
        _p.unlink()
        print(f"  Deleted stale cache: {_old}")

# ── Configuration ─────────────────────────────────────────────────────────
DATA_DIR    = "/content/drive/MyDrive/DroneRF/DroneRF"
OUTPUT_CSV  = "dronerf_features_v5.csv"
DB_PATH     = "rf_fingerprint_db.json"
RANDOM_SEED = 42

# FIX-1: window 1024 → 8192
WINDOW_SIZE           = 8192
STEP_SIZE             = 4096
TARGET_TOTAL          = 4500
MAX_SEGMENTS_PER_FILE = 50   # fewer segments per file because window is larger
FS                    = 10e6

# FIX-3: gate 0.65 → 0.50
CONFIDENCE_THRESHOLD   = 0.50
HIGH_THREAT_THRESHOLD  = 0.85
TRUST_MIN_OBSERVATIONS = 10   # slightly lower — larger windows give richer signal
TRUST_MAX_VARIANCE     = 0.20
SIMILARITY_THRESHOLD   = 0.82
THREAT_FLOOR           = 0.50

# FIX-5: auto-classify confidence threshold
AUTO_CLASSIFY_CONF     = 0.50

ANOMALY_WEIGHTS = {"mahal": 0.35, "gmm": 0.30, "isoforest": 0.25, "lr": 0.10}

DL_HIDDEN       = (256, 128, 64)
DL_DROPOUT      = 0.30
DL_LR           = 1e-3
DL_WEIGHT_DECAY = 1e-4
DL_BATCH_SIZE   = 128
DL_MAX_EPOCHS   = 200
DL_ES_PATIENCE  = 15
DL_LR_PATIENCE  = 5
DL_LR_FACTOR    = 0.5
DL_LR_MIN       = 1e-6

# ── Install & Import ──────────────────────────────────────────────────────
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "numpy", "pandas", "scipy", "scikit-learn",
     "imbalanced-learn", "matplotlib", "seaborn", "tqdm", "torch", "-q"],
    check=True,
)

import gc, os, re, time, warnings, hashlib, json, copy
from concurrent.futures import ProcessPoolExecutor, as_completed
from collections        import defaultdict, deque, Counter
from dataclasses        import dataclass, field
from pathlib            import Path
from typing             import Dict, List, Optional, Tuple

import numpy             as np
import pandas            as pd
import matplotlib;       matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn           as sns
from matplotlib.patches  import Patch
from tqdm                import tqdm

import torch
import torch.nn             as nn
import torch.optim          as optim
from torch.utils.data       import DataLoader, TensorDataset

from scipy.stats  import kurtosis, skew
from scipy.signal import hilbert, welch, stft

from sklearn.calibration       import CalibratedClassifierCV
from sklearn.decomposition     import PCA
from sklearn.ensemble          import (RandomForestClassifier,
                                        GradientBoostingClassifier,
                                        IsolationForest)
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model      import LogisticRegression   # FIX-6: replaces GNB
from sklearn.metrics           import (accuracy_score, f1_score,
                                        precision_score, recall_score,
                                        classification_report,
                                        confusion_matrix, log_loss)
from sklearn.mixture           import GaussianMixture, BayesianGaussianMixture
from sklearn.model_selection   import (train_test_split, StratifiedKFold,
                                        cross_val_score)
from sklearn.naive_bayes       import GaussianNB
from sklearn.neural_network    import MLPClassifier
from sklearn.preprocessing     import RobustScaler
from sklearn.svm               import SVC
from imblearn.over_sampling    import SMOTE

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Imports ready  |  device={DEVICE}")

# ── Class labels ─────────────────────────────────────────────────────────
CLASS_NAMES = {0: "Background RF activities", 1: "AR drone", 2: "Phantom drone"}
N_CLASSES   = len(CLASS_NAMES)
BG_NAME     = CLASS_NAMES[0]

FOLDER_MAP = {
    "background": 0, "ar drone": 1, "ar_drone": 1, "ardrone": 1, "phantom": 2,
}
BUI_MAP = {
    "00000": 0,
    "10000": 1, "10001": 1, "10010": 1, "10011": 1,
    "10100": 1, "10101": 1, "10110": 1,
    "11000": 2, "11001": 2, "11010": 2,
}


def folder_to_class(name: str) -> Optional[int]:
    n = name.lower().strip()
    for k, v in FOLDER_MAP.items():
        if k in n:
            return v
    return None


def bui_to_class(fname: str) -> Optional[int]:
    m = re.search(r"\d{5}", Path(fname).stem)
    return BUI_MAP.get(m.group(0)) if m else None


def discover_files(data_dir: str) -> Dict[int, List[Path]]:
    root = Path(data_dir)
    cf   = defaultdict(list)
    if not root.exists():
        raise FileNotFoundError(f"Not found: {data_dir}")
    for subdir in sorted(root.iterdir()):
        if not subdir.is_dir():
            continue
        c = folder_to_class(subdir.name)
        if c is None:
            continue
        files = sorted(subdir.rglob("*.csv"))
        if files:
            cf[c].extend(files)
    if cf:
        for c, fl in sorted(cf.items()):
            print(f"  [{c}] {CLASS_NAMES[c]:<35} {len(fl):>4} files")
        return dict(cf)
    for fp in sorted(root.rglob("*.csv")):
        c = bui_to_class(fp.name)
        if c is not None:
            cf[c].append(fp)
    if not cf:
        raise RuntimeError("No CSV files found — check DATA_DIR")
    for c, fl in sorted(cf.items()):
        print(f"  [{c}] {CLASS_NAMES[c]:<35} {len(fl):>4} files")
    return dict(cf)


def read_dronerf_real(filepath: Path) -> np.ndarray:
    try:
        df   = pd.read_csv(filepath, header=None, dtype=np.float32)
        flat = df.values.ravel()
        if len(flat) == 0:
            raise ValueError("Empty file")
        return flat
    except Exception as e:
        print(f"  [WARN] {filepath.name}: {e}")
        return np.zeros(WINDOW_SIZE * 2, dtype=np.float32)


# ── Feature names ─────────────────────────────────────────────────────────
FEATURE_NAMES = [
    "amp_mean", "amp_std", "amp_var", "amp_min", "amp_max",
    "amp_range", "amp_kurtosis", "amp_skew",
    "env_mean", "env_std", "env_min", "env_max", "signal_power_db", "IQ_corr",
    "peak_freq_hz", "bandwidth_hz", "spectral_entropy",
    "spectral_centroid", "spectral_spread", "spectral_rolloff_85",
    "psd_mean_db", "psd_max_db",
    "ifreq_mean", "ifreq_std", "ifreq_range", "ifreq_kurtosis",
    "energy_band1", "energy_band2", "energy_band3", "energy_band4",
    "stft_flux_var", "stft_sub1_var", "stft_sub2_var",
    "stft_sub3_var", "stft_sub4_var",
    "spec_kurtosis", "spec_skewness", "l_kurtosis",
    "spec_flatness", "stft_entropy",
    "iq_power_ratio", "am_depth", "crest_factor",
    "phase_jitter", "iq_corr_sq",
    "acf_short", "acf_medium", "acf_long", "acf_ratio",
    "spec_asymmetry", "kurt_entropy_product", "snr_like_db",
]
N_FEATURES = len(FEATURE_NAMES)
assert N_FEATURES == 52
FIDX = {name: i for i, name in enumerate(FEATURE_NAMES)}
print(f"✓ Feature pipeline v5: {N_FEATURES} features  |  window={WINDOW_SIZE}")


# ════════════════════════════════════════════════════════════════════════════
# FEATURE EXTRACTION
# ════════════════════════════════════════════════════════════════════════════

def _pearson(x: np.ndarray, y: np.ndarray) -> float:
    xm = x - x.mean(); ym = y - y.mean()
    return float(np.dot(xm, ym) / ((np.dot(xm,xm)*np.dot(ym,ym))**0.5 + 1e-12))

def _pct(arr: np.ndarray, q: float) -> float:
    c = arr[np.isfinite(arr)]
    return float(np.percentile(c, q)) if len(c) > 0 else 0.0


def extract_features(real_seg: np.ndarray, fs: float = FS) -> np.ndarray:
    real     = real_seg.astype(np.float64)
    N        = len(real)
    analytic = hilbert(real)
    I        = analytic.real
    Q        = analytic.imag
    envelope = np.abs(analytic)
    out      = np.empty(52, dtype=np.float32)
    pos      = 0

    # GROUP A: amplitude envelope (8)
    amp_mean = float(envelope.mean())
    amp_std  = float(envelope.std())
    amp_min  = float(envelope.min())
    amp_max  = float(envelope.max())
    amp_ptp  = amp_max - amp_min
    amp_kurt = float(kurtosis(envelope)) if amp_std > 1e-8 else 0.0
    amp_skew = float(skew(envelope))     if amp_std > 1e-8 else 0.0
    out[pos:pos+8] = [amp_mean, amp_std, amp_std**2, amp_min, amp_max,
                      amp_ptp, amp_kurt, amp_skew]
    pos += 8

    # GROUP B: envelope / IQ (6)
    env_mean = float(envelope.mean())
    env_std  = float(envelope.std())
    power_db = float(10.0 * np.log10(np.dot(envelope, envelope) / N + 1e-12))
    iq_corr  = _pearson(I, Q) if env_std > 1e-12 else 0.0
    out[pos:pos+6] = [env_mean, env_std, float(envelope.min()),
                      float(envelope.max()), power_db, iq_corr]
    pos += 6

    # GROUP C: Welch PSD (8)
    nperseg = min(512, N // 4)    # FIX-1 benefit: larger window → finer frequency resolution
    fw, psd = welch(envelope, fs=fs, nperseg=nperseg,
                    noverlap=nperseg//2, return_onesided=True)
    pa      = np.clip(np.abs(psd), 1e-12, None)
    pa_sum  = pa.sum()
    pd_db   = 10.0 * np.log10(pa)
    pk      = int(pa.argmax())
    above   = fw[pd_db > pd_db[pk] - 10.0]
    bw      = float(above.max() - above.min()) if len(above) > 1 else 0.0
    pn      = pa / pa_sum
    entropy = float(-np.dot(pn, np.log2(pn + 1e-12)))
    cen     = float(np.dot(fw, pa) / pa_sum)
    spread  = float(np.sqrt(np.dot((fw - cen)**2, pa) / pa_sum))
    cs      = np.cumsum(pa)
    rol     = min(int(np.searchsorted(cs, 0.85*cs[-1])), len(fw)-1)
    out[pos:pos+8] = [fw[pk], bw, entropy, cen, spread,
                      fw[rol], float(pd_db.mean()), float(pd_db.max())]
    pos += 8

    # GROUP D: instantaneous frequency (4)
    ifreq = np.diff(np.unwrap(np.angle(analytic)))
    if len(ifreq) >= 2 and ifreq.std() > 1e-8:
        ifreq_mean = float(ifreq.mean())
        ifreq_std  = float(ifreq.std())
        ifreq_ptp  = float(ifreq.max() - ifreq.min())   # NumPy 2.0 safe
        ifreq_kurt = float(kurtosis(ifreq))
    else:
        ifreq_mean = ifreq_std = ifreq_ptp = ifreq_kurt = 0.0
    out[pos:pos+4] = [ifreq_mean, ifreq_std, ifreq_ptp, ifreq_kurt]
    pos += 4

    # GROUP E: band energy ratios (4)
    q_sz = max(1, len(pa)//4)
    out[pos:pos+4] = [pa[:q_sz].sum()/pa_sum, pa[q_sz:2*q_sz].sum()/pa_sum,
                      pa[2*q_sz:3*q_sz].sum()/pa_sum, pa[3*q_sz:].sum()/pa_sum]
    pos += 4

    # GROUP F: STFT sub-band (5)
    stft_np   = min(128, N//4)   # larger STFT window from larger segment
    _, _, Zxx = stft(envelope, fs=fs, nperseg=stft_np,
                     noverlap=stft_np//2, return_onesided=True)
    Sxx       = np.abs(Zxx)**2 + 1e-12
    fm        = Sxx.mean(0)
    out[pos]  = float(np.diff(fm).var())
    bsz       = max(1, Sxx.shape[0]//4)
    for b in range(4):
        out[pos+1+b] = float(Sxx[b*bsz:(b+1)*bsz,:].mean(0).var())
    pos += 5

    # GROUP G: higher-order spectral (5)
    pa_s      = np.sort(pa)
    spec_kurt = float(kurtosis(pa))
    spec_skew = float(skew(pa))
    L2 = pa_s[1::2].mean() - pa_s[::2].mean()
    L4 = (pa_s[3::4].mean() - 3.0*pa_s[2::4].mean()
          + 3.0*pa_s[1::4].mean() - pa_s[::4].mean())
    l_kurt    = float(L4 / (L2 + 1e-12))
    spec_flat = float(np.exp(np.log(pa+1e-12).mean() - np.log(pa.mean()+1e-12)))
    Sxx_n     = Sxx.mean(1); Sxx_n /= (Sxx_n.sum() + 1e-12)
    stft_ent  = float(-np.dot(Sxx_n, np.log2(Sxx_n+1e-12)))
    out[pos:pos+5] = [spec_kurt, spec_skew, l_kurt, spec_flat, stft_ent]
    pos += 5

    # GROUP H: IQ modulation proxies (5)
    I_pow = float(np.dot(I,I)/N); Q_pow = float(np.dot(Q,Q)/N)
    rms   = float((np.dot(envelope,envelope)/N)**0.5)
    out[pos:pos+5] = [I_pow/(Q_pow+1e-12),
                      float((envelope.max()-envelope.min())/(env_mean+1e-12)),
                      float(envelope.max()/(rms+1e-12)),
                      float(np.diff(ifreq).std()) if len(ifreq)>=2 else 0.0,
                      iq_corr**2]
    pos += 5

    # GROUP I: cyclostationary (4)
    en    = ((envelope-env_mean)/(env_std+1e-12))[:min(512,N)]
    N_s   = len(en)
    ls, lm, ll = min(50,N_s//10), min(200,N_s//3), min(400,N_s//2)
    acf_s = _pearson(en[:N_s-ls], en[ls:])
    acf_m = _pearson(en[:N_s-lm], en[lm:])
    acf_l = _pearson(en[:N_s-ll], en[ll:])
    out[pos:pos+4] = [acf_s, acf_m, acf_l, acf_m/(acf_s+1e-6)]
    pos += 4

    # GROUP J: spectral shape (3)
    spec_asym = float((pa[fw>=cen].sum()-pa[fw<cen].sum())/(pa_sum+1e-12))
    kurt_ent  = amp_kurt * entropy
    top10 = pa_s[::-1][:max(1,len(pa_s)//10)].mean()
    bot50 = pa_s[::-1][len(pa_s)//2:].mean()
    out[pos:pos+3] = [spec_asym, kurt_ent,
                      float(10.0*np.log10(top10/(bot50+1e-12)))]

    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def safe_extract(seg: np.ndarray, fs: float = FS) -> np.ndarray:
    try:
        return extract_features(seg, fs)
    except Exception as e:
        safe_extract._n = getattr(safe_extract, "_n", 0) + 1
        if safe_extract._n <= 3:
            print(f"  [safe_extract #{safe_extract._n}]: {e}")
        return np.zeros(N_FEATURES, dtype=np.float32)


# ════════════════════════════════════════════════════════════════════════════
# DATASET BUILDER — FIX-2: skip segment-0, start at WINDOW_SIZE
# ════════════════════════════════════════════════════════════════════════════

def _worker_extract_file(args: tuple):
    filepath, cls_int, quota = args
    eff   = min(quota, MAX_SEGMENTS_PER_FILE)
    feats = []
    try:
        raw = read_dronerf_real(Path(filepath))
        n   = len(raw)
        # FIX-2: skip first window (zero-pad transient from SDR ramp-up)
        start = WINDOW_SIZE
        while start + WINDOW_SIZE <= n and len(feats) < eff:
            feats.append(safe_extract(raw[start:start+WINDOW_SIZE]))
            start += STEP_SIZE
    except Exception as e:
        print(f"  skip {Path(filepath).name}: {e}")
    if not feats:
        return cls_int, filepath, np.empty((0, N_FEATURES), np.float32)
    return cls_int, filepath, np.vstack(feats).astype(np.float32)


def _try_load_cache(output_csv: str) -> Optional[pd.DataFrame]:
    p = Path(output_csv)
    if not p.exists():
        return None
    try:
        df = pd.read_csv(output_csv)
    except Exception:
        p.unlink(missing_ok=True); return None
    feat_cols = [c for c in df.columns if c in FEATURE_NAMES]
    if len(feat_cols) != N_FEATURES:
        p.unlink(missing_ok=True); return None
    for col in ["amp_std", "env_std"]:
        if col in df.columns and df[col].var() < 1e-4:
            print(f"  Cache degenerate — rebuilding.")
            p.unlink(missing_ok=True); return None
    print(f"  Cache OK: {len(df):,} rows  (amp_std var={df['amp_std'].var():.4f})")
    return df


def build_or_load_dataset(data_dir: str, output_csv: str = OUTPUT_CSV) -> pd.DataFrame:
    cached = _try_load_cache(output_csv)
    if cached is not None:
        print(f"⚡ Cache → {output_csv}"); return cached

    print(f"\n{'='*60}\nBUILDING DATASET  window={WINDOW_SIZE}  step={STEP_SIZE}\n"
          f"  (FIX-2: segment-0 skipped per file)\n{'='*60}")
    cf       = discover_files(data_dir)
    n_cls    = len(cf)
    q_base   = TARGET_TOTAL // n_cls
    leftover = TARGET_TOTAL - q_base * n_cls
    rng      = np.random.default_rng(RANDOM_SEED)

    work_items = []
    for i, (cls_int, flist) in enumerate(sorted(cf.items())):
        quota     = q_base + (1 if i < leftover else 0)
        shuffled  = list(flist); rng.shuffle(shuffled)
        remaining = quota
        for fp in shuffled:
            if remaining <= 0: break
            per_file = min(remaining, MAX_SEGMENTS_PER_FILE)
            work_items.append((str(fp), cls_int, per_file))
            remaining -= per_file

    n_workers = max(1, min(os.cpu_count() or 1, len(work_items)))
    results   = []
    try:
        with ProcessPoolExecutor(max_workers=n_workers) as pool:
            futures = {pool.submit(_worker_extract_file, item): item
                       for item in work_items}
            for fut in tqdm(as_completed(futures), total=len(futures), desc="Extracting"):
                try: results.append(fut.result())
                except Exception as e: print(f"  Worker error: {e}")
    except Exception as e:
        print(f"  Parallel failed ({e}), serial fallback…")
        results = [_worker_extract_file(item) for item in tqdm(work_items)]

    class_counts: Dict[int,int] = defaultdict(int)
    mats, cls_ints, fnames = [], [], []
    for cls_int, fpath, mat in sorted(results):
        if mat.shape[0] == 0: continue
        mats.append(mat); cls_ints.extend([cls_int]*mat.shape[0])
        fnames.extend([Path(fpath).name]*mat.shape[0])
        class_counts[cls_int] += mat.shape[0]

    if not mats:
        raise RuntimeError("No features extracted. Check DATA_DIR.")

    X_all  = np.vstack(mats).astype(np.float32)
    labels = np.array(cls_ints, dtype=np.int32)

    amp_var = X_all[:, FIDX["amp_std"]].var()
    print(f"\n  Sanity: amp_std var={amp_var:.4f}  "
          f"{'✓' if amp_var > 1e-4 else '❌ STILL ZERO'}")

    df = pd.DataFrame(X_all, columns=FEATURE_NAMES)
    df.insert(0, "label_int",   labels)
    df.insert(1, "label_name",  [CLASS_NAMES[c] for c in labels])
    df.insert(2, "source_file", fnames)
    df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    df.to_csv(output_csv, index=False)
    for c, n in sorted(class_counts.items()):
        print(f"  ✓ {CLASS_NAMES[c]}: {n} segments")
    print(f"✓ Saved {len(df):,} rows → {output_csv}")
    return df


# ════════════════════════════════════════════════════════════════════════════
# DATA PREPARATION
# ════════════════════════════════════════════════════════════════════════════

def prepare_data(df: pd.DataFrame):
    X_all = np.nan_to_num(
        df[FEATURE_NAMES].fillna(0).values.astype(np.float32),
        nan=0., posinf=0., neginf=0.)
    y_all = df["label_int"].values.astype(np.int64)
    counts        = {c: int((y_all==c).sum()) for c in np.unique(y_all)}
    valid_classes = [c for c, n in counts.items() if n >= 6]
    mask          = np.isin(y_all, valid_classes)
    X_use, y_use  = X_all[mask], y_all[mask]
    lmap            = {old: new for new, old in enumerate(sorted(valid_classes))}
    y_mapped        = np.array([lmap[yi] for yi in y_use], dtype=np.int64)
    CLASSES_PRESENT = [CLASS_NAMES[c] for c in sorted(valid_classes)]
    N_CLS           = len(CLASSES_PRESENT)
    print(f"\n  Classes: {N_CLS}")
    for i, cn in enumerate(CLASSES_PRESENT):
        print(f"    [{i}] {cn}  ({(y_mapped==i).sum()} samples)")
    return X_use, y_mapped, lmap, CLASSES_PRESENT, N_CLS


# ════════════════════════════════════════════════════════════════════════════
# FEATURE VALIDATION  — FIX-7: use all 52 features, no MI cutoff
# ════════════════════════════════════════════════════════════════════════════

def validate_and_select_features(X: np.ndarray, y: np.ndarray):
    print(f"\n{'='*60}\nFEATURE VALIDATION + SELECTION (v5: all 52)\n{'='*60}")
    sc  = RobustScaler()
    X_s = sc.fit_transform(X)
    X_s = np.nan_to_num(X_s, nan=0., posinf=0., neginf=0.)

    var_mask = X_s.var(0) > 1e-15
    n_drop   = int((~var_mask).sum())
    print(f"  After RobustScaler: dropped={n_drop}  kept={var_mask.sum()}")
    if n_drop == N_FEATURES:
        raise RuntimeError("All features zero variance — check extraction.")

    if var_mask.sum() >= 3:
        pca = PCA(n_components=min(10, var_mask.sum()))
        pca.fit(X_s[:, var_mask])
        cum_var = np.cumsum(pca.explained_variance_ratio_)
        print("  PCA variance:")
        for k in [3, 5, 10]:
            k2 = min(k, len(cum_var))
            vd = "✓ GOOD" if cum_var[k2-1]>0.60 else "△ MARGINAL" if cum_var[k2-1]>0.40 else "✗ POOR"
            print(f"    Top-{k2:>2} PCs: {cum_var[k2-1]:.3f}  [{vd}]")

    mi      = mutual_info_classif(X_s, y, random_state=RANDOM_SEED)
    top_idx = np.argsort(mi)[::-1]
    print(f"\n  Top-15 features by MI:")
    for rank, i in enumerate(top_idx[:15], 1):
        s = "★★" if mi[i]>0.30 else "★" if mi[i]>0.10 else "○" if mi[i]>0.05 else "△"
        print(f"    {rank:>2}. {FEATURE_NAMES[i]:<30}  {mi[i]:.4f}  {s}")

    print(f"\n  Best MI = {mi[top_idx[0]]:.4f}")
    print(f"\n  FIX-7: Using ALL {N_FEATURES} features (no MI cutoff)")
    print(f"         MI cutoff loses spectral shape features needed for AR vs Phantom")

    # FIX-7: use all 52, sorted by MI for interpretability
    selected_idx = top_idx   # all features, sorted by MI descending
    scaler_sel   = RobustScaler()
    X_sel        = scaler_sel.fit_transform(X[:, selected_idx])
    X_sel        = np.nan_to_num(X_sel, nan=0., posinf=0., neginf=0.)
    return X_sel, selected_idx, scaler_sel, mi


# ════════════════════════════════════════════════════════════════════════════
# DEEP RF NET
# ════════════════════════════════════════════════════════════════════════════

class DeepRFNet(nn.Module):
    def __init__(self, in_features, n_classes, hidden=DL_HIDDEN, dropout=DL_DROPOUT):
        super().__init__()
        layers = []; prev = in_features
        for h in hidden:
            layers += [nn.Linear(prev,h), nn.BatchNorm1d(h),
                       nn.ReLU(inplace=True), nn.Dropout(p=dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x): return self.net(x)


def train_deep_rf_net(X_tr, y_tr, X_val, y_val, n_classes):
    model     = DeepRFNet(X_tr.shape[1], n_classes).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=DL_LR, weight_decay=DL_WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=DL_LR_PATIENCE,
        factor=DL_LR_FACTOR, min_lr=DL_LR_MIN)
    criterion = nn.CrossEntropyLoss()
    Xtr_t = torch.from_numpy(X_tr.astype(np.float32)).to(DEVICE)
    ytr_t = torch.from_numpy(y_tr.astype(np.int64)).to(DEVICE)
    Xva_t = torch.from_numpy(X_val.astype(np.float32)).to(DEVICE)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t),
                        batch_size=DL_BATCH_SIZE, shuffle=True, drop_last=False)
    best_f1, best_weights, es_counter = -1., copy.deepcopy(model.state_dict()), 0
    train_losses, val_f1s = [], []
    for epoch in range(1, DL_MAX_EPOCHS+1):
        model.train(); epoch_loss = 0.
        for Xb, yb in loader:
            optimizer.zero_grad(); loss = criterion(model(Xb), yb)
            loss.backward(); optimizer.step()
            epoch_loss += loss.item()*len(Xb)
        epoch_loss /= len(Xtr_t); train_losses.append(epoch_loss)
        model.eval()
        with torch.no_grad():
            preds = model(Xva_t).argmax(1).cpu().numpy()
        val_f1 = float(f1_score(y_val, preds, average="macro", zero_division=0))
        val_f1s.append(val_f1); scheduler.step(val_f1)
        if val_f1 > best_f1 + 1e-5:
            best_f1, best_weights, es_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            es_counter += 1
            if es_counter >= DL_ES_PATIENCE:
                print(f"    Early stop ep={epoch}  best F1={best_f1:.4f}"); break
        if epoch % 20 == 0 or epoch == 1:
            print(f"    Ep {epoch:>3}  loss={epoch_loss:.4f}  val_F1={val_f1:.4f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")
    model.load_state_dict(best_weights); model.eval()
    print(f"  ✓ DeepRFNet best val F1={best_f1:.4f}")
    return model, train_losses, val_f1s


class TorchWrapper:
    def __init__(self, model, n_classes):
        self.model = model; self.n_classes = n_classes
    def predict_proba(self, X):
        self.model.eval()
        with torch.no_grad():
            return torch.softmax(
                self.model(torch.from_numpy(X.astype(np.float32)).to(DEVICE)),
                dim=1).cpu().numpy()
    def predict(self, X): return self.predict_proba(X).argmax(1)


# ════════════════════════════════════════════════════════════════════════════
# MODEL TRAINING — FIX-6: LR replaces GNB as lightweight ensemble partner
# ════════════════════════════════════════════════════════════════════════════

def build_and_evaluate(X_sel, y, CLASSES_PRESENT):
    print(f"\n{'='*60}\nMODEL TRAINING\n{'='*60}")
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_sel, y, test_size=0.20, stratify=y, random_state=RANDOM_SEED)
    _, X_val, _, y_val = train_test_split(
        X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=RANDOM_SEED)
    _, cnts    = np.unique(y_tr, return_counts=True)
    k_smote    = max(1, min(5, int(cnts.min())-1))
    X_sm, y_sm = SMOTE(random_state=RANDOM_SEED, k_neighbors=k_smote).fit_resample(X_tr, y_tr)
    print(f"  SMOTE k={k_smote}: {X_sm.shape[0]:,} train | "
          f"Val: {X_val.shape[0]:,} | Test: {X_te.shape[0]:,}")
    n_cls = len(CLASSES_PRESENT)

    # RF
    rf = RandomForestClassifier(500, class_weight="balanced", max_features="sqrt",
                                 min_samples_leaf=2, random_state=RANDOM_SEED,
                                 n_jobs=-1, oob_score=True)
    rf.fit(X_sm, y_sm)
    yp_rf  = rf.predict(X_te)
    acc_rf = accuracy_score(y_te, yp_rf)
    f1_rf  = f1_score(y_te, yp_rf, average="macro", zero_division=0)
    print(f"\n  [A] RF:  acc={acc_rf:.4f}  F1={f1_rf:.4f}  OOB={rf.oob_score_:.4f}")

    # FIX-6: Logistic Regression as lightweight ensemble partner (replaces GNB)
    # Use Pipeline with StandardScaler — LR converges much better with scaled features
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler as SS
    lr_clf = Pipeline([
        ("sc", SS()),
        ("lr", LogisticRegression(C=1.0, class_weight="balanced",
                                   max_iter=2000, random_state=RANDOM_SEED,
                                   solver="lbfgs", multi_class="multinomial")),
    ])
    lr_clf.fit(X_sm, y_sm)
    yp_lr  = lr_clf.predict(X_te)
    acc_lr = accuracy_score(y_te, yp_lr)
    f1_lr  = f1_score(y_te, yp_lr, average="macro", zero_division=0)
    print(f"  [B] LR:  acc={acc_lr:.4f}  F1={f1_lr:.4f}  (replaces GNB — 0.03ms infer)")

    # GBT
    gbt = GradientBoostingClassifier(n_estimators=200, learning_rate=0.08,
                                      max_depth=5, subsample=0.8,
                                      min_samples_leaf=5, random_state=RANDOM_SEED)
    t0 = time.time(); gbt.fit(X_sm, y_sm); t_gbt = round(time.time()-t0, 2)
    yp_gbt  = gbt.predict(X_te)
    acc_gbt = accuracy_score(y_te, yp_gbt)
    f1_gbt  = f1_score(y_te, yp_gbt, average="macro", zero_division=0)
    print(f"  [C] GBT: acc={acc_gbt:.4f}  F1={f1_gbt:.4f}  ({t_gbt}s)")

    # DeepRFNet
    print(f"\n  [D] DeepRFNet:")
    deep_model, dl_loss, dl_f1s = train_deep_rf_net(X_sm, y_sm, X_val, y_val, n_cls)
    dw       = TorchWrapper(deep_model, n_cls)
    yp_deep  = dw.predict(X_te)
    acc_deep = accuracy_score(y_te, yp_deep)
    f1_deep  = f1_score(y_te, yp_deep, average="macro", zero_division=0)
    print(f"  [D] DeepRFNet: acc={acc_deep:.4f}  F1={f1_deep:.4f}")

    print(f"\n  Classification report (RF):")
    print(classification_report(y_te, yp_rf, target_names=CLASSES_PRESENT, zero_division=0))

    return (rf, lr_clf, gbt, dw,
            X_te, y_te, X_val, y_val, X_sm, y_sm, yp_rf,
            acc_rf, f1_rf, acc_lr, f1_lr, acc_deep, f1_deep,
            k_smote, dl_loss, dl_f1s)


def run_supplementary_benchmark(X_sm, y_sm, X_te, y_te, CLASSES_PRESENT, N_CLS, yp_rf):
    print(f"\n{'='*65}\nSUPPLEMENTARY BENCHMARK\n{'='*65}")
    bench = {}
    def run_bench(name, model, cv=True):
        t0 = time.time(); model.fit(X_sm, y_sm); tt = round(time.time()-t0, 3)
        yp = model.predict(X_te)
        acc = round(accuracy_score(y_te, yp), 4)
        f1  = round(f1_score(y_te, yp, average="macro", zero_division=0), 4)
        cvf = (round(cross_val_score(
                   model, X_sm, y_sm,
                   cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_SEED),
                   scoring="f1_macro", n_jobs=-1).mean(), 4) if cv else None)
        bench[name] = {"accuracy":acc,"f1_macro":f1,"cv_f1":cvf,"train_s":tt}
        print(f"  {name:<28} acc={acc:.4f}  f1={f1:.4f}  t={tt}s  cv={cvf}")
    run_bench("GradientBoosting_100",
        GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                    max_depth=4, random_state=RANDOM_SEED))
    run_bench("SVM_RBF",
        SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced",
            probability=True, random_state=RANDOM_SEED))
    run_bench("LogisticRegression",
        LogisticRegression(C=1.0, class_weight="balanced",
                            max_iter=1000, random_state=RANDOM_SEED))
    run_bench("MLP_sklearn",
        MLPClassifier(hidden_layer_sizes=(256,128,64), max_iter=300,
                      early_stopping=True, validation_fraction=0.15,
                      learning_rate_init=1e-3, alpha=1e-4,
                      solver="adam", random_state=RANDOM_SEED))
    pd.DataFrame([{"Model":k,**v} for k,v in bench.items()]
                 ).sort_values("f1_macro",ascending=False).to_csv("benchmark_results.csv",index=False)
    print("  ✓ benchmark_results.csv saved")
    print("\nFull RF classification report:")
    print(classification_report(y_te, yp_rf, target_names=CLASSES_PRESENT, zero_division=0))
    return bench


# ════════════════════════════════════════════════════════════════════════════
# ANOMALY DETECTION
# ════════════════════════════════════════════════════════════════════════════

class MahalanobisDetector:
    def fit(self, X, y):
        self.params = {}
        for c in np.unique(y):
            Xc = X[y==c]; mu = Xc.mean(0)
            cov = np.cov(Xc, rowvar=False) + np.eye(Xc.shape[1])*1e-2
            try:    prec = np.linalg.inv(cov)
            except: prec = np.linalg.pinv(cov)
            self.params[c] = (mu, prec)
        self.threshold = _pct(self.score(X), 99); return self
    def score(self, X):
        dists = []
        for mu, prec in self.params.values():
            d = X-mu; dists.append(np.sqrt(np.maximum(np.einsum("ni,ij,nj->n",d,prec,d),0.)))
        return np.nan_to_num(np.stack(dists,1).min(1), nan=0.,posinf=0.,neginf=0.)

class GMMDetector:
    def fit(self, X, y):
        self.gmms = {}
        for c in np.unique(y):
            Xc = X[y == c]
            nc = min(3, max(1, len(Xc) // 10))
            # Try progressively safer covariance configs until one succeeds
            fitted = False
            for cov_type, reg in [("diag", 0.1), ("full", 0.1),
                                   ("spherical", 0.1), ("spherical", 1.0)]:
                try:
                    self.gmms[c] = GaussianMixture(
                        n_components=nc, covariance_type=cov_type,
                        random_state=RANDOM_SEED, reg_covar=reg,
                        max_iter=300).fit(Xc)
                    fitted = True
                    break
                except (ValueError, np.linalg.LinAlgError):
                    continue
            if not fitted:
                # Absolute fallback: 1-component spherical with high regularisation
                self.gmms[c] = GaussianMixture(
                    n_components=1, covariance_type="spherical",
                    random_state=RANDOM_SEED, reg_covar=10.0,
                    max_iter=300).fit(Xc)
        self.threshold = _pct(self.score(X), 99)
        return self

    def score(self, X):
        return np.nan_to_num(
            -np.stack([g.score_samples(X) for g in self.gmms.values()], 1).max(1),
            nan=0., posinf=0., neginf=0.)

class IsoForestDetector:
    def fit(self, X, y=None):
        self.model = IsolationForest(n_estimators=200,contamination=0.02,
                                      n_jobs=-1,random_state=RANDOM_SEED).fit(X)
        self.threshold = _pct(self.score(X), 99); return self
    def score(self, X):
        return np.nan_to_num(-self.model.score_samples(X), nan=0.,posinf=0.,neginf=0.)

class LRLikelihoodDetector:
    """FIX-6: uses LogisticRegression log-prob instead of GNB.
    Uses its own StandardScaler since LR is sensitive to feature scale."""
    def fit(self, X, y):
        from sklearn.preprocessing import StandardScaler
        from sklearn.pipeline import Pipeline
        self.lr = Pipeline([
            ("sc", StandardScaler()),
            ("lr", LogisticRegression(C=1.0, class_weight="balanced",
                                       max_iter=2000, random_state=RANDOM_SEED,
                                       solver="lbfgs"))
        ]).fit(X, y)
        self.threshold = _pct(self.score(X), 99)
        return self

    def score(self, X):
        return np.nan_to_num(-self.lr.predict_log_proba(X).max(1),
                              nan=0., posinf=0., neginf=0.)


def train_anomaly_detectors(X_sm, y_sm):
    print(f"\n{'='*65}\nANOMALY DETECTION\n{'='*65}")
    det_mahal  = MahalanobisDetector().fit(X_sm, y_sm)
    det_gmm    = GMMDetector().fit(X_sm, y_sm)
    det_iso    = IsoForestDetector().fit(X_sm)
    det_lr_ll  = LRLikelihoodDetector().fit(X_sm, y_sm)
    print("  ✓ All 4 anomaly detectors trained (Mahal, GMM, IsoForest, LR-LL)")
    return det_mahal, det_gmm, det_iso, det_lr_ll


class ThreatScorer:
    def __init__(self, det_mahal, det_gmm, det_iso, det_lr_ll, X_sm, anomaly_weights):
        self._dets  = [("mahal",det_mahal),("gmm",det_gmm),
                       ("isoforest",det_iso),("lr",det_lr_ll)]
        self._wvals = [anomaly_weights[n] for n,_ in self._dets]
        self._lo, self._hi = [], []
        for _, det in self._dets:
            s=det.score(X_sm); lo=_pct(s,1); hi=_pct(s,99)
            if hi<=lo: hi=lo+1.0
            self._lo.append(lo); self._hi.append(hi)
        self._norm_params = {name:(lo,hi)
                              for (name,_),lo,hi in zip(self._dets,self._lo,self._hi)}
        raw_thr        = _pct(self.compute(X_sm), 97)
        self.threshold = max(float(raw_thr), THREAT_FLOOR)
        print(f"  Threat threshold: {self.threshold:.4f}")

    def compute(self, X_sc):
        n = X_sc.shape[0] if X_sc.ndim>1 else 1
        result = np.zeros(n, dtype=np.float64)
        for (_,det),w,lo,hi in zip(self._dets,self._wvals,self._lo,self._hi):
            result += w * np.clip((det.score(X_sc)-lo)/(hi-lo+1e-12), 0., 1.)
        return result


def make_bayesian_fn(rf_clf, lr_clf, det_iso, threat_scorer, CLASSES_PRESENT, N_CLS):
    """FIX-6: Bayesian fusion uses RF × LR (geometric mean) instead of RF × GNB."""
    iso_lo = threat_scorer._norm_params["isoforest"][0]
    iso_hi = threat_scorer._norm_params["isoforest"][1]

    def bayesian_confidence_breakdown(X_sc, ts):
        eps = 1e-12; n = N_CLS
        rf_p  = rf_clf.predict_proba(X_sc)[0].astype(np.float64)+eps; rf_p/=rf_p.sum()
        lr_p  = lr_clf.predict_proba(X_sc)[0].astype(np.float64)+eps; lr_p/=lr_p.sum()
        combined = np.sqrt(rf_p*lr_p); combined/=combined.sum()
        winner_idx = int(combined.argmax()); sorted_c = np.sort(combined)[::-1]
        cum = np.cumsum(sorted_c)
        norm_H    = float(-np.dot(combined,np.log(combined+eps))/(np.log(n)+eps))
        iso_norm  = float(np.clip((det_iso.score(X_sc)-iso_lo)/(iso_hi-iso_lo+1e-12),0,1)[0])
        epistemic = float(np.clip(0.5*iso_norm+0.5*norm_H,0.,1.))
        aleatoric = float(np.clip(norm_H*(1.-ts),0.,1.))
        margin    = float(sorted_c[0]-sorted_c[1]) if n>1 else 1.
        calibrated= float(sorted_c[0]*(1.+margin)/2.)
        return {
            "winner":                CLASSES_PRESENT[winner_idx],
            "posterior_probs":       {CLASSES_PRESENT[i]:round(float(combined[i]),4) for i in range(n)},
            "max_confidence":        round(float(sorted_c[0]),4),
            "predictive_entropy":    round(norm_H,4),
            "epistemic_uncertainty": round(epistemic,4),
            "aleatoric_uncertainty": round(aleatoric,4),
            "margin":                round(margin,4),
            "calibrated_confidence": round(calibrated,4),
            "n_credible_classes":    int(np.searchsorted(cum,0.95))+1,
            "threat_score":          round(float(ts),4),
            "is_novel":              bool(ts>0.5 or norm_H>0.85 or margin<0.10),
            "lr_probs":              {CLASSES_PRESENT[i]:round(float(lr_p[i]),4) for i in range(n)},
        }
    return bayesian_confidence_breakdown


def build_online_lr_curve(X_sm, y_sm, X_te, y_te, CLASSES_PRESENT, n_batches=20):
    """Online learning curve using SGDClassifier (LogisticRegression equivalent with partial_fit)."""
    from sklearn.linear_model import SGDClassifier
    print(f"\n{'='*60}\nONLINE SGD (LR) LEARNING CURVE\n{'='*60}")
    online = SGDClassifier(loss="log_loss", alpha=1e-4, class_weight="balanced",
                            random_state=RANDOM_SEED, max_iter=1)
    f1_curve = []; classes = np.arange(len(CLASSES_PRESENT))
    for b, idx in enumerate(np.array_split(np.random.permutation(len(X_sm)), n_batches), 1):
        online.partial_fit(X_sm[idx], y_sm[idx], classes=classes)
        f1 = round(f1_score(online.predict(X_te), y_te, average="macro", zero_division=0), 4)
        f1_curve.append(f1)
        if b%5==0 or b==n_batches: print(f"  Batch {b:>2}/{n_batches}  F1={f1:.4f}")
    print(f"  Final online SGD F1: {f1_curve[-1]:.4f}")
    return online, f1_curve


# ════════════════════════════════════════════════════════════════════════════
# TEMPORAL TRACKER + FINGERPRINT DB
# FIX-5: auto-classify on promotion
# FIX-6: dynamic DB with class labels
# ════════════════════════════════════════════════════════════════════════════

def emitter_hash(fv: np.ndarray, n_bins: int = 50) -> str:
    qfp = np.round(np.clip(fv, -10., 10.)*n_bins).astype(np.int32)
    return hashlib.md5(qfp.tobytes()).hexdigest()[:16]


@dataclass
class EmitterRecord:
    emitter_id:      str
    feature_history: deque = field(default_factory=lambda: deque(maxlen=50))
    first_seen:      float = field(default_factory=time.time)
    last_seen:       float = field(default_factory=time.time)
    seen_count:      int   = 0
    threat_scores:   List[float] = field(default_factory=list)
    trust_score:     float = 0.0
    promoted:        bool  = False
    auto_class:      Optional[str] = None   # FIX-5
    auto_conf:       float = 0.0            # FIX-5

    def update(self, fv, ts):
        self.feature_history.append(fv.copy()); self.last_seen=time.time()
        self.seen_count+=1; self.threat_scores.append(float(ts))

    @property
    def mean_features(self):
        return np.mean(np.stack(list(self.feature_history)), 0)

    @property
    def feature_variance(self):
        if len(self.feature_history)<2: return 1.0
        return float(np.mean(np.var(np.stack(list(self.feature_history)),0)))

    @property
    def mean_threat(self):
        return float(np.mean(self.threat_scores)) if self.threat_scores else 1.0

    def compute_trust(self):
        obs_t  = float(1./(1.+np.exp(-(self.seen_count-TRUST_MIN_OBSERVATIONS)/3.)))
        stab_t = float(max(0.,1.-self.feature_variance/(TRUST_MAX_VARIANCE+1e-9)))
        safe_t = float(max(0.,1.-self.mean_threat))
        vals   = [obs_t, stab_t, safe_t]
        self.trust_score = float(np.clip(len(vals)/sum(1./(v+1e-9) for v in vals),0.,1.))
        return self.trust_score

    def is_trustworthy(self):
        return (self.seen_count>=TRUST_MIN_OBSERVATIONS and
                self.feature_variance<=TRUST_MAX_VARIANCE and
                self.mean_threat<HIGH_THREAT_THRESHOLD)


class TemporalTracker:
    def __init__(self): self.registry={}; self.total_obs=0

    def observe(self, fv, ts):
        eid = emitter_hash(fv)
        if eid not in self.registry:
            self.registry[eid] = EmitterRecord(emitter_id=eid)
        rec = self.registry[eid]
        rec.update(fv, ts); rec.compute_trust(); self.total_obs+=1
        return rec

    def summary(self):
        n   = len(self.registry)
        nt  = sum(1 for r in self.registry.values() if r.is_trustworthy())
        nth = sum(1 for r in self.registry.values() if r.mean_threat>=HIGH_THREAT_THRESHOLD)
        return f"Tracker: {n} emitters | trustworthy={nt} threat={nth} monitor={n-nt-nth}"


def cosine_sim(a, b):
    a=a.ravel().astype(np.float64); b=b.ravel().astype(np.float64)
    return float(np.dot(a,b)/((np.dot(a,a)*np.dot(b,b))**0.5+1e-12))


class FingerprintDatabase:
    """
    FIX-6: Dynamic trusted DB.
    Each entry stores: fingerprint, label (auto-classified or SAFE_UNKNOWN),
    predicted_class, confidence, seen_count, first_seen, last_updated.
    Entries upgrade from SAFE_UNKNOWN → identified class when classifier
    confidence rises above AUTO_CLASSIFY_CONF.
    """
    def __init__(self, path):
        self.path=path; self.trusted={}; self.suspicious={}; self._load()

    def _load(self):
        if Path(self.path).exists():
            try:
                data=json.load(open(self.path))
                self.trusted=data.get("trusted",{}); self.suspicious=data.get("suspicious",{})
                print(f"  Loaded RF DB: {len(self.trusted)} trusted, {len(self.suspicious)} suspicious")
            except: print("  RF DB corrupted — starting fresh")
        else: print("  RF DB: starting fresh")

    def save(self):
        json.dump({"trusted":self.trusted,"suspicious":self.suspicious},
                  open(self.path,"w"), indent=2)

    def match(self, fv):
        best_sim, best_id, best_store = -1., None, ""
        for sname, db in (("trusted",self.trusted),("suspicious",self.suspicious)):
            for eid, rec in db.items():
                sim=cosine_sim(fv, np.array(rec["fingerprint"]))
                if sim>best_sim: best_sim,best_id,best_store=sim,eid,sname
        return best_id, float(best_sim), best_store

    def add_trusted(self, eid: str, fv: np.ndarray, seen_count: int,
                    predicted_class: str, confidence: float) -> None:
        """
        FIX-5 + FIX-6: store auto-classified label.
        If confidence >= AUTO_CLASSIFY_CONF → use predicted_class.
        Otherwise → SAFE_UNKNOWN_xxx.
        """
        is_new = eid not in self.trusted

        if confidence >= AUTO_CLASSIFY_CONF and predicted_class != BG_NAME:
            label = f"AUTO_{predicted_class.upper().replace(' ','_')}"
        else:
            label = f"SAFE_UNKNOWN_{len(self.trusted)+1:03d}" if is_new else \
                    self.trusted[eid]["label"]

        entry = {
            "fingerprint":     fv.tolist(),
            "label":           label,
            "predicted_class": predicted_class,
            "confidence":      round(confidence, 4),
            "seen_count":      seen_count,
            "first_seen":      self.trusted[eid]["first_seen"] if not is_new else time.time(),
            "last_updated":    time.time(),
        }
        self.trusted[eid] = entry
        self.save()
        if is_new:
            print(f"  ✅  PROMOTED → {label}  "
                  f"(class={predicted_class}, conf={confidence:.2f}, seen={seen_count})")
        else:
            print(f"  🔄  UPDATED → {label}  "
                  f"(class={predicted_class}, conf={confidence:.2f})")

    def add_suspicious(self, eid, fv, seen_count=0):
        if eid not in self.suspicious:
            self.suspicious[eid] = {
                "fingerprint": fv.tolist(),
                "label": f"THREAT_{len(self.suspicious)+1:03d}",
                "seen_count": seen_count, "added_at": time.time()}
        else: self.suspicious[eid]["seen_count"] = seen_count
        self.save()

    def upgrade_entry(self, eid: str, predicted_class: str,
                       confidence: float, fv: np.ndarray, seen_count: int) -> None:
        """FIX-6: Re-run classification on existing trusted emitter to upgrade label."""
        if eid not in self.trusted: return
        current_label = self.trusted[eid]["label"]
        if (confidence >= AUTO_CLASSIFY_CONF and
                predicted_class != BG_NAME and
                "SAFE_UNKNOWN" in current_label):
            self.add_trusted(eid, fv, seen_count, predicted_class, confidence)

    def summary(self):
        return f"RF Database: {len(self.trusted)} trusted | {len(self.suspicious)} suspicious"

    def trusted_summary(self) -> str:
        if not self.trusted: return "  (empty)"
        lines = []
        for eid, rec in self.trusted.items():
            lines.append(f"  {eid[:8]}.. → {rec['label']:<35} "
                         f"conf={rec.get('confidence',0):.2f}  "
                         f"seen={rec.get('seen_count',0)}")
        return "\n".join(lines)


ICONS = {
    "FRIENDLY_DRONE":"🟢", "BACKGROUND":"⚪", "POTENTIAL_THREAT":"🔴",
    "CONFIRMED_THREAT":"🚨", "SAFE_NEW_DRONE":"🔵", "TRUSTED_NEW_DRONE":"🔷",
    "UNKNOWN_MONITOR":"🟡", "AUTO_AR_DRONE":"🟩", "AUTO_PHANTOM_DRONE":"🟦",
}


# ════════════════════════════════════════════════════════════════════════════
# DECISION LOGIC — FIX-3, FIX-5, FIX-6
# ════════════════════════════════════════════════════════════════════════════

def make_classify_fn(rf_clf, lr_clf, scaler_sel, selected_idx,
                      threat_scorer, bayesian_fn, fp_db, temporal_tracker,
                      CLASSES_PRESENT, THREAT_THRESHOLD):
    def classify_signal(fv_raw, return_bayes=True):
        fv_raw = np.nan_to_num(fv_raw.astype(np.float32), nan=0.,posinf=0.,neginf=0.)
        fv_sel = fv_raw[selected_idx] if len(fv_raw)==N_FEATURES else fv_raw
        X_sc   = np.nan_to_num(scaler_sel.transform(fv_sel.reshape(1,-1)),
                               nan=0.,posinf=0.,neginf=0.)
        eid    = emitter_hash(fv_raw)
        ts     = float(threat_scorer.compute(X_sc)[0])
        bayes  = bayesian_fn(X_sc, ts) if return_bayes else {}
        result = {"label":None,"bayesian":bayes,"emitter_id":eid,
                  "trust_score":0.,"promoted":False,"auto_class":None}

        if bayes:
            conf   = bayes.get("calibrated_confidence", 0.)
            winner = bayes.get("winner", "")
        else:
            combined = np.sqrt(rf_clf.predict_proba(X_sc)[0] *
                                lr_clf.predict_proba(X_sc)[0] + 1e-12)
            combined /= combined.sum()
            conf   = float(combined.max())
            winner = CLASSES_PRESENT[int(combined.argmax())]

        # FIX-3: gate lowered to 0.50
        if conf >= CONFIDENCE_THRESHOLD:
            result["label"] = "BACKGROUND" if winner == BG_NAME else "FRIENDLY_DRONE"
            return result

        # DB match check
        match_id, sim, store = fp_db.match(fv_raw)
        if sim >= SIMILARITY_THRESHOLD and store == "trusted":
            db_entry = fp_db.trusted[match_id]
            db_label = db_entry.get("label", "TRUSTED_NEW_DRONE")
            # Return the auto-classified label if available
            if db_label.startswith("AUTO_"):
                result["label"] = db_label
            else:
                result["label"] = "TRUSTED_NEW_DRONE"
            if bayes:
                bayes["db_match"]      = db_label
                bayes["db_similarity"] = round(float(sim), 4)
            return result

        # Track emitter
        rec = temporal_tracker.observe(fv_raw, ts)
        result["trust_score"] = float(rec.trust_score)
        result["emitter_id"]  = rec.emitter_id

        # Threat path
        if ts >= THREAT_THRESHOLD or rec.mean_threat >= HIGH_THREAT_THRESHOLD:
            fp_db.add_suspicious(rec.emitter_id, rec.mean_features, rec.seen_count)
            result["label"] = ("CONFIRMED_THREAT" if rec.seen_count>=3
                                else "POTENTIAL_THREAT")
            return result

        # FIX-5: auto-classify on promotion
        if rec.is_trustworthy() and not rec.promoted:
            # Run classifier on the emitter's mean feature vector
            mean_sel = rec.mean_features[selected_idx] \
                if len(rec.mean_features)==N_FEATURES else rec.mean_features
            X_mean   = np.nan_to_num(scaler_sel.transform(mean_sel.reshape(1,-1)),
                                     nan=0.,posinf=0.,neginf=0.)
            rf_p  = rf_clf.predict_proba(X_mean)[0] + 1e-12
            lr_p  = lr_clf.predict_proba(X_mean)[0] + 1e-12
            comb  = np.sqrt(rf_p*lr_p); comb /= comb.sum()
            ac    = CLASSES_PRESENT[int(comb.argmax())]
            ac_conf = float(comb.max())

            fp_db.add_trusted(rec.emitter_id, rec.mean_features,
                               rec.seen_count, ac, ac_conf)
            rec.promoted      = True
            rec.auto_class    = ac
            rec.auto_conf     = ac_conf
            result["promoted"]   = True
            result["auto_class"] = ac

            # Return auto-classified label
            if ac_conf >= AUTO_CLASSIFY_CONF and ac != BG_NAME:
                result["label"] = f"AUTO_{ac.upper().replace(' ','_')}"
            else:
                result["label"] = "SAFE_NEW_DRONE"
            return result

        # FIX-6: try to upgrade existing trusted entry
        if rec.emitter_id in fp_db.trusted and rec.seen_count % 5 == 0:
            mean_sel = rec.mean_features[selected_idx] \
                if len(rec.mean_features)==N_FEATURES else rec.mean_features
            X_mean   = np.nan_to_num(scaler_sel.transform(mean_sel.reshape(1,-1)),
                                     nan=0.,posinf=0.,neginf=0.)
            rf_p  = rf_clf.predict_proba(X_mean)[0] + 1e-12
            lr_p  = lr_clf.predict_proba(X_mean)[0] + 1e-12
            comb  = np.sqrt(rf_p*lr_p); comb /= comb.sum()
            fp_db.upgrade_entry(rec.emitter_id, CLASSES_PRESENT[int(comb.argmax())],
                                float(comb.max()), rec.mean_features, rec.seen_count)

        if rec.promoted or rec.emitter_id in fp_db.trusted:
            db_label = fp_db.trusted.get(rec.emitter_id, {}).get("label", "SAFE_NEW_DRONE")
            result["label"] = db_label if db_label.startswith("AUTO_") else "SAFE_NEW_DRONE"
        else:
            result["label"] = "UNKNOWN_MONITOR"
        return result
    return classify_signal


# ════════════════════════════════════════════════════════════════════════════
# SYNTHETIC SIMULATION — FIX-4: raw-space profiles
# ════════════════════════════════════════════════════════════════════════════

def compute_synthetic_profiles(df: pd.DataFrame) -> dict:
    """
    FIX-4: compute synthetic profile values from actual CSV data.
    Profiles are now in raw feature space (not normalised z-values).
    Each profile offsets from the class mean by multiples of the class std.
    """
    medians = df.groupby("label_int")[FEATURE_NAMES].median()
    stds    = df.groupby("label_int")[FEATURE_NAMES].std()

    # AR drone stats (class 1)
    ar_pow_med   = float(medians.loc[1, "signal_power_db"])    if 1 in medians.index else -15.0
    ar_pow_std   = float(stds.loc[1,    "signal_power_db"])    if 1 in stds.index    else 5.0
    ar_ent_med   = float(medians.loc[1, "spectral_entropy"])   if 1 in medians.index else 5.5
    ar_ent_std   = float(stds.loc[1,    "spectral_entropy"])   if 1 in stds.index    else 0.8
    ar_bw_med    = float(medians.loc[1, "bandwidth_hz"])       if 1 in medians.index else 2e6
    ar_bw_std    = float(stds.loc[1,    "bandwidth_hz"])       if 1 in stds.index    else 0.5e6

    # Background stats (class 0)
    bg_pow_med   = float(medians.loc[0, "signal_power_db"])    if 0 in medians.index else -35.0
    bg_ent_med   = float(medians.loc[0, "spectral_entropy"])   if 0 in medians.index else 6.5

    print(f"\n  Synthetic profile calibration (FIX-4):")
    print(f"    AR drone: power={ar_pow_med:.1f}dB (±{ar_pow_std:.1f})  "
          f"entropy={ar_ent_med:.2f} (±{ar_ent_std:.2f})")
    print(f"    Background: power={bg_pow_med:.1f}dB  entropy={bg_ent_med:.2f}")

    return {
        "DJI_Neo_Threat": {
            "signal_power_db":   ar_pow_med + 2.5 * ar_pow_std,   # high power
            "spectral_entropy":  ar_ent_med - 2.0 * ar_ent_std,   # low entropy (narrowband)
            "bandwidth_hz":      ar_bw_med  + 2.5 * ar_bw_std,    # wide bandwidth
            "noise_scale_frac":  0.15,   # 15% noise relative to IQR
            "is_threat": True, "seed": 3001, "note": "OcuSync3 / wideband erratic",
        },
        "Autel_EVO3_Threat": {
            "signal_power_db":   ar_pow_med + 1.8 * ar_pow_std,
            "spectral_entropy":  ar_ent_med - 2.5 * ar_ent_std,
            "bandwidth_hz":      ar_bw_med  + 1.8 * ar_bw_std,
            "noise_scale_frac":  0.15,
            "is_threat": True, "seed": 3002, "note": "Aggressive FHSS",
        },
        "Harmless_Surveyor": {
            "signal_power_db":   ar_pow_med - 1.0 * ar_pow_std,   # low power
            "spectral_entropy":  ar_ent_med + 1.5 * ar_ent_std,   # higher entropy
            "bandwidth_hz":      ar_bw_med  - 1.5 * ar_bw_std,    # narrow
            "noise_scale_frac":  0.05,
            "is_threat": False, "seed": 3003, "note": "Stable narrowband surveyor",
        },
        "Delivery_Bot": {
            "signal_power_db":   ar_pow_med - 0.8 * ar_pow_std,
            "spectral_entropy":  ar_ent_med + 1.2 * ar_ent_std,
            "bandwidth_hz":      ar_bw_med  - 1.2 * ar_bw_std,
            "noise_scale_frac":  0.05,
            "is_threat": False, "seed": 3004, "note": "Urban delivery drone",
        },
    }


def generate_synthetic_obs_v5(prof: dict, df: pd.DataFrame, n: int) -> List[np.ndarray]:
    """FIX-4: generate in raw feature space using actual per-feature medians."""
    rng = np.random.default_rng(prof["seed"])
    # Base: mean of all training data (neutral starting point)
    base = df[FEATURE_NAMES].median().values.copy().astype(np.float64)
    # Override the 3 key discriminative features
    for feat, key in [("signal_power_db","signal_power_db"),
                       ("spectral_entropy","spectral_entropy"),
                       ("bandwidth_hz","bandwidth_hz")]:
        if key in prof:
            base[FIDX[feat]] = prof[key]
    # Noise proportional to the per-feature std of training data
    feat_stds = df[FEATURE_NAMES].std().values.astype(np.float64)
    noise_std = feat_stds * prof["noise_scale_frac"]
    return [(base + rng.standard_normal(N_FEATURES)*noise_std).astype(np.float32)
            for _ in range(n)]


def run_synthetic_simulation(classify_signal, df, TRUST_MIN_OBSERVATIONS):
    N_OBS    = TRUST_MIN_OBSERVATIONS + 5
    profiles = compute_synthetic_profiles(df)
    print(f"\n{'='*65}\nSYNTHETIC SIMULATION  ({N_OBS} obs each)  [FIX-4: raw-space profiles]\n{'='*65}")
    sim_results = {}
    for name, prof in profiles.items():
        print(f"\n── {name}  [{prof['note']}]")
        decisions = []
        for step, fv in enumerate(generate_synthetic_obs_v5(prof, df, N_OBS), 1):
            dec  = classify_signal(fv, return_bayes=True); decisions.append(dec)
            b    = dec["bayesian"]; promo = " ← PROMOTED" if dec.get("promoted") else ""
            ac   = f" [{dec.get('auto_class','')}]" if dec.get("auto_class") else ""
            print(f"  t={step:>2}  {ICONS.get(dec['label'],'❓')} "
                  f"{dec['label']:<28}  "
                  f"conf={b.get('calibrated_confidence',0):.3f}  "
                  f"ts={b.get('threat_score',0):.3f}  "
                  f"trust={dec['trust_score']:.3f}{promo}{ac}")
        sim_results[name] = decisions
        final = decisions[-1]
        print(f"  FINAL: {ICONS.get(final['label'],'?')} {final['label']}  "
              f"conf={final['bayesian'].get('calibrated_confidence',0):.3f}")
    return sim_results


def realtime_detection_loop(csv_path, classify_signal, max_segments=15, verbose=True):
    real_all  = read_dronerf_real(Path(csv_path)); decisions = []
    if verbose:
        bui   = bui_to_class(Path(csv_path).name)
        bname = CLASS_NAMES.get(bui, "Unknown")
        sep   = "─"*80
        print(f"\n{sep}\nREAL-TIME SCAN: {Path(csv_path).name}  [{bname}]\n{sep}")
        print(f"{'Seg':>4}  {'Label':<28}  {'Conf':>8}  {'TScore':>7}  {'Ep':>7}  {'Margin':>7}")
        print(sep)
    n_total = len(real_all)
    # FIX-2: skip segment-0 in real-time too
    for seg_idx in range(1, max_segments+1):
        start = seg_idx * STEP_SIZE; end = start + WINDOW_SIZE
        if end > n_total: break
        fv  = safe_extract(real_all[start:end])
        res = classify_signal(fv, return_bayes=True); decisions.append(res)
        if verbose:
            b = res["bayesian"]
            lbl = res["label"] or "None"
            print(f"{seg_idx:>4}  {ICONS.get(lbl,'❓')} {lbl:<26}  "
                  f"{b.get('calibrated_confidence',0):>8.4f}  "
                  f"{b.get('threat_score',0):>7.4f}  "
                  f"{b.get('epistemic_uncertainty',0):>7.4f}  "
                  f"{b.get('margin',0):>7.4f}")
    if verbose and decisions:
        lc = Counter(d["label"] for d in decisions)
        print("─"*80)
        print(f"SUMMARY ({len(decisions)} segs): " +
              "  ".join(f"{ICONS.get(k,'?')} {k}: {v}" for k,v in sorted(lc.items())))
        n_thr = sum(1 for d in decisions if "THREAT" in d.get("label",""))
        if n_thr: print(f"⚠️  ALERT — {n_thr} threat segment(s) detected!")
    return decisions


def run_full_evaluation(X_te, y_te, scaler_sel, selected_idx,
                         classify_signal, CLASSES_PRESENT):
    print(f"\n{'='*65}\nFULL SYSTEM EVALUATION\n{'='*65}")
    X_te_raw = scaler_sel.inverse_transform(X_te)
    X_52     = np.zeros((len(X_te_raw), N_FEATURES), dtype=np.float32)
    for sp, oc in enumerate(selected_idx):
        X_52[:, oc] = X_te_raw[:, sp].astype(np.float32)

    test_decs = []
    for i in range(len(X_52)):
        dec = classify_signal(X_52[i], return_bayes=True)
        dec["true_class"] = CLASSES_PRESENT[y_te[i]]; test_decs.append(dec)

    test_df = pd.DataFrame(test_decs)
    for col in ["calibrated_confidence","predictive_entropy","epistemic_uncertainty",
                "aleatoric_uncertainty","margin","threat_score","n_credible_classes",
                "max_confidence","is_novel","winner"]:
        test_df[col] = test_df["bayesian"].apply(
            lambda b,c=col: b.get(c) if isinstance(b,dict) else None)

    known_mask  = ~test_df["label"].isin(
        ["POTENTIAL_THREAT","CONFIRMED_THREAT","UNKNOWN_MONITOR",
         "SAFE_NEW_DRONE","TRUSTED_NEW_DRONE"])
    correct     = ((test_df.loc[known_mask,"winner"]==
                    test_df.loc[known_mask,"true_class"]).mean()
                   if known_mask.sum() else 0.)
    false_alarm = test_df["label"].isin(["POTENTIAL_THREAT","CONFIRMED_THREAT"]).mean()
    bg_recall   = (test_df[test_df["true_class"]==BG_NAME]["label"]
                   .eq("BACKGROUND").mean()
                   if (test_df["true_class"]==BG_NAME).any() else 0.)

    print(f"  Test set size        : {len(test_df):,}")
    print(f"  Known-drone accuracy : {correct:.2%}")
    print(f"  False alarm rate     : {false_alarm:.2%}")
    print(f"  Background recall    : {bg_recall:.2%}")
    print("\n  Label distribution:")
    for lbl, cnt in test_df["label"].value_counts().items():
        pct = cnt/len(test_df)
        print(f"    {ICONS.get(lbl,'?')} {lbl:<30} {cnt:>5}  ({pct:.1%})")

    test_df.to_csv("system_test_decisions.csv", index=False)
    print("  ✓ system_test_decisions.csv")
    return test_df, known_mask, correct, false_alarm, bg_recall


def make_dashboard(X_sel, y_mapped, mi, rf_clf, X_te, y_te,
                   test_df, known_mask, f1_curve, bench_results, yp_rf,
                   CLASSES_PRESENT, acc_rf, f1_rf, acc_lr, f1_lr, acc_deep, f1_deep,
                   sim_results, THREAT_THRESHOLD, dl_loss, dl_f1s, selected_idx):
    C={"friendly":"#1D9E75","threat":"#D85A30","background":"#888780",
       "safe_new":"#3B82F6","monitor":"#F59E0B","trusted_new":"#6366F1",
       "bayesian":"#7F77DD","deep":"#E11D48","auto":"#5DCAA5"}
    importances   = rf_clf.feature_importances_
    n_imp         = len(importances)
    feature_names = [FEATURE_NAMES[selected_idx[i]] for i in range(n_imp)]

    fig = plt.figure(figsize=(28,28))
    gs  = gridspec.GridSpec(3,3,figure=fig,hspace=0.58,wspace=0.42)

    ax1 = fig.add_subplot(gs[0,:2])
    top_n=min(20,n_imp); ti=np.argsort(importances)[::-1][:top_n]; ti_nm=[feature_names[i] for i in ti]
    pcols=["#7F77DD" if "energy" in nm or "band" in nm else
            "#3B82F6" if any(k in nm for k in ("freq","bandwidth","entropy","centroid","spread","rolloff","psd","stft","spec")) else
            "#1D9E75" for nm in ti_nm]
    ax1.barh(ti_nm[::-1],importances[ti][::-1],color=pcols[::-1],height=0.72)
    ax1.set_xlabel("Feature importance (Gini)",fontsize=11)
    ax1.set_title("Top Feature Importances — RF v5 (all 52, Hilbert IQ)",fontsize=12,fontweight="500")
    ax1.legend(handles=[Patch(facecolor="#1D9E75",label="Amplitude/IQ"),
                         Patch(facecolor="#3B82F6",label="Spectral/STFT"),
                         Patch(facecolor="#7F77DD",label="Band energy")],fontsize=9)
    ax1.tick_params(labelsize=9)

    ax2 = fig.add_subplot(gs[0,2])
    for lbl,col,a in [("FRIENDLY_DRONE",C["friendly"],0.8),("BACKGROUND",C["background"],0.6)]:
        vals=test_df.loc[test_df["label"]==lbl,"calibrated_confidence"].dropna()
        if len(vals): ax2.hist(vals,bins=25,alpha=a,density=True,color=col,label=lbl)
    ax2.axvline(CONFIDENCE_THRESHOLD,color="black",ls="--",lw=2,label=f"Gate={CONFIDENCE_THRESHOLD}")
    ax2.set_title("Confidence Distribution (gate=0.50)",fontsize=11,fontweight="500"); ax2.legend(fontsize=8)

    ax3 = fig.add_subplot(gs[1,0])
    for lbl,col in [("FRIENDLY_DRONE",C["friendly"]),("BACKGROUND",C["background"]),
                     ("POTENTIAL_THREAT",C["threat"]),("CONFIRMED_THREAT",C["threat"]),
                     ("SAFE_NEW_DRONE",C["safe_new"]),("UNKNOWN_MONITOR",C["monitor"]),
                     ("AUTO_AR_DRONE",C["auto"]),("AUTO_PHANTOM_DRONE",C["trusted_new"])]:
        m=test_df["label"]==lbl
        if m.sum()>0: ax3.scatter(test_df.loc[m,"aleatoric_uncertainty"],
                                   test_df.loc[m,"epistemic_uncertainty"],
                                   c=col,alpha=0.4,s=12,label=f"{lbl}(n={m.sum()})")
    ax3.set_xlabel("Aleatoric"); ax3.set_ylabel("Epistemic")
    ax3.set_title("Uncertainty Decomposition",fontsize=11,fontweight="500"); ax3.legend(fontsize=7)

    ax4 = fig.add_subplot(gs[1,1])
    if "Harmless_Surveyor" in sim_results:
        sims=sim_results["Harmless_Surveyor"]
        ax4.plot(range(1,len(sims)+1),[d["trust_score"] for d in sims],
                 color=C["safe_new"],lw=2.5,marker="o",ms=6,label="Trust")
        ax4.plot(range(1,len(sims)+1),
                 [d["bayesian"].get("calibrated_confidence",0) for d in sims],
                 color=C["bayesian"],lw=2,ls="--",label="Conf")
        ax4.axhline(0.5,color="gray",ls=":"); ax4.set_ylim(0,1.05)
        ax4.set_title("Trust Accumulation + Auto-classify",fontsize=11,fontweight="500")
        ax4.legend(fontsize=8)

    ax5 = fig.add_subplot(gs[1,2])
    if "DJI_Neo_Threat" in sim_results:
        sims=sim_results["DJI_Neo_Threat"]
        ax5.plot(range(1,len(sims)+1),[d["bayesian"].get("threat_score",0) for d in sims],
                 color=C["threat"],lw=2.5,marker="s",ms=6,label="Threat score")
        ax5.axhline(THREAT_THRESHOLD,color="black",ls="--",lw=1.5,
                    label=f"Thr={THREAT_THRESHOLD:.3f}")
        ax5.set_ylim(0,1.05); ax5.set_title("Threat Detection (DJI_Neo_Threat)",fontsize=11,fontweight="500")
        ax5.legend(fontsize=8)

    ax6 = fig.add_subplot(gs[2,0])
    if dl_loss and dl_f1s:
        ep=range(1,len(dl_loss)+1); ax6t=ax6.twinx()
        ax6.plot(ep,dl_loss,color=C["deep"],lw=2,label="Train loss")
        ax6t.plot(ep,dl_f1s,color=C["bayesian"],lw=2,ls="--",label="Val F1")
        ax6.set_title("DeepRFNet Training",fontsize=11,fontweight="500")
        l1,lb1=ax6.get_legend_handles_labels(); l2,lb2=ax6t.get_legend_handles_labels()
        ax6.legend(l1+l2,lb1+lb2,fontsize=8)

    ax7 = fig.add_subplot(gs[2,1])
    known_s = test_df[known_mask & test_df["winner"].notna()].copy()
    if len(known_s)>0:
        pres=sorted(set(known_s["true_class"])|set(known_s["winner"]))
        cm=confusion_matrix(known_s["true_class"],known_s["winner"],labels=pres)
        sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=pres,yticklabels=pres,
                    ax=ax7,cbar=False,annot_kws={"size":9})
        ax7.set_title("Confusion Matrix (RF+LR ensemble)",fontsize=11,fontweight="500")
        ax7.tick_params(labelsize=8)

    ax8 = fig.add_subplot(gs[2,2])
    if f1_curve:
        ax8.plot(range(1,len(f1_curve)+1),f1_curve,color=C["bayesian"],lw=2.5,marker="o",ms=4)
        ax8.axhline(max(f1_curve),color="gray",ls="--",lw=1,
                    label=f"Max F1={max(f1_curve):.4f}")
        ax8.set_ylim(0,1.05); ax8.set_title("Online SGD Learning Curve",fontsize=11,fontweight="500")
        ax8.legend(fontsize=8)

    fig.suptitle(
        "Continual RF Drone Monitoring System v5\n"
        "FIX-1: window×8  |  FIX-2: skip seg-0  |  FIX-3: gate=0.50  |  "
        "FIX-4: raw profiles  |  FIX-5: auto-classify  |  FIX-6: dynamic DB  |  FIX-7: all 52 features",
        fontsize=11,fontweight="600")
    plt.savefig("continual_monitor_dashboard.png",dpi=150,bbox_inches="tight")
    plt.close()
    print("✓ Dashboard → continual_monitor_dashboard.png")


# ════════════════════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("\n" + "█"*65)
    print("  CONTINUAL RF DRONE MONITORING v5")
    print("  7 fixes applied — see header for full changelog")
    print("█"*65)

    # 1. Dataset
    df = build_or_load_dataset(DATA_DIR)
    X_use, y_mapped, lmap, CLASSES_PRESENT, N_CLS = prepare_data(df)

    # 2. Feature validation + FIX-7: all 52 selected
    X_sel, selected_idx, scaler_sel, mi = validate_and_select_features(
        X_use, y_mapped)

    # 3. Train
    (rf_clf, lr_clf, gbt_clf, deep_wrapper,
     X_te, y_te, X_val, y_val, X_sm, y_sm, yp_rf,
     acc_rf, f1_rf, acc_lr, f1_lr, acc_deep, f1_deep,
     k_smote, dl_loss, dl_f1s) = build_and_evaluate(X_sel, y_mapped, CLASSES_PRESENT)

    # 4. Online learning curve
    online_sgd, f1_curve = build_online_lr_curve(
        X_sm, y_sm, X_te, y_te, CLASSES_PRESENT)

    # 5. Supplementary benchmark
    bench_results = run_supplementary_benchmark(
        X_sm, y_sm, X_te, y_te, CLASSES_PRESENT, N_CLS, yp_rf)

    # 6. Anomaly detection
    det_mahal, det_gmm, det_iso, det_lr_ll = train_anomaly_detectors(X_sm, y_sm)
    threat_scorer    = ThreatScorer(det_mahal, det_gmm, det_iso, det_lr_ll,
                                     X_sm, ANOMALY_WEIGHTS)
    THREAT_THRESHOLD = threat_scorer.threshold

    # 7. Bayesian fusion + decision logic
    bayesian_fn      = make_bayesian_fn(rf_clf, lr_clf, det_iso,
                                        threat_scorer, CLASSES_PRESENT, N_CLS)
    fp_db            = FingerprintDatabase(DB_PATH)
    temporal_tracker = TemporalTracker()
    classify_signal  = make_classify_fn(
        rf_clf, lr_clf, scaler_sel, selected_idx,
        threat_scorer, bayesian_fn, fp_db, temporal_tracker,
        CLASSES_PRESENT, THREAT_THRESHOLD)
    print(f"\n✓ {fp_db.summary()}")

    # 8. Synthetic simulation — FIX-4: raw-space profiles
    sim_results = run_synthetic_simulation(
        classify_signal, df, TRUST_MIN_OBSERVATIONS)
    print(f"\n{temporal_tracker.summary()}")

    # 9. Print trusted DB state after simulation
    print(f"\n{'='*60}\nTRUSTED FINGERPRINT DATABASE STATE\n{'='*60}")
    print(fp_db.trusted_summary())

    # 10. Real-time demo
    print(f"\n{'='*65}\nREAL-TIME DETECTION DEMO (FIX-2: seg-0 skipped)\n{'='*65}")
    try:
        _demo_files = []
        for _flist in discover_files(DATA_DIR).values():
            _demo_files.extend(_flist[:1])
        for _fp in _demo_files[:3]:
            realtime_detection_loop(str(_fp), classify_signal, max_segments=8)
    except Exception as _e:
        print(f"  (Demo skipped: {_e})")

    # 11. Full evaluation
    test_df, known_mask, correct, false_alarm, bg_recall = run_full_evaluation(
        X_te, y_te, scaler_sel, selected_idx, classify_signal, CLASSES_PRESENT)

    # 12. Dashboard
    make_dashboard(X_sel, y_mapped, mi, rf_clf, X_te, y_te,
                   test_df, known_mask, f1_curve, bench_results, yp_rf,
                   CLASSES_PRESENT, acc_rf, f1_rf, acc_lr, f1_lr, acc_deep, f1_deep,
                   sim_results, THREAT_THRESHOLD, dl_loss, dl_f1s, selected_idx)
    try:
        from google.colab import files
        files.download("continual_monitor_dashboard.png")
    except Exception: pass

    # 13. Save
    fp_db.save()
    pd.DataFrame({"batch":range(1,len(f1_curve)+1),"f1_macro":f1_curve}
                 ).to_csv("online_sgd_learning_curve.csv",index=False)
    pd.DataFrame({"epoch":range(1,len(dl_loss)+1),"train_loss":dl_loss,"val_f1":dl_f1s}
                 ).to_csv("deeprfnet_training_curve.csv",index=False)

    print(f"\n{'='*72}")
    print("CONTINUAL RF DRONE MONITORING v5 — FINAL SUMMARY")
    print(f"{'='*72}")
    print(f"""
7 FIXES APPLIED:
  FIX-1  window 1024→8192 samples (0.10ms→0.82ms) — captures full signal cycles
  FIX-2  skip segment-0 zero-pad transient (SDR ramp-up) in training AND inference
  FIX-3  confidence gate 0.65→0.50 — borderline drones enter trust/threat path
  FIX-4  synthetic profiles in raw feature space (computed from actual CSV medians)
  FIX-5  auto-classify unknown emitters on promotion via RF+LR ensemble
  FIX-6  dynamic trusted DB — entries upgrade SAFE_UNKNOWN → AUTO_AR_DRONE etc.
  FIX-7  all 52 features used (no MI cutoff) — keeps spectral shape features

DATASET   : {len(df):,} segments  |  window={WINDOW_SIZE}  step={STEP_SIZE}  |  {N_CLS} classes
MODELS
  RF       : acc={acc_rf:.4f}  F1={f1_rf:.4f}  OOB={rf_clf.oob_score_:.4f}
  LR       : acc={acc_lr:.4f}  F1={f1_lr:.4f}  (0.03ms inference — replaces GNB)
  DeepRFNet: acc={acc_deep:.4f}  F1={f1_deep:.4f}
EVALUATION
  Known-drone accuracy : {correct:.2%}
  False alarm rate     : {false_alarm:.2%}
  Background recall    : {bg_recall:.2%}
TRUSTED DB
{fp_db.trusted_summary()}
""")
    print("="*72)
    print("System ready.")

✓ Imports ready  |  device=cpu
✓ Feature pipeline v5: 52 features  |  window=8192

█████████████████████████████████████████████████████████████████
  CONTINUAL RF DRONE MONITORING v5
  7 fixes applied — see header for full changelog
█████████████████████████████████████████████████████████████████

BUILDING DATASET  window=8192  step=4096
  (FIX-2: segment-0 skipped per file)
  [0] Background RF activities              82 files
  [1] AR drone                             162 files
  [2] Phantom drone                         42 files


Extracting: 100%|██████████| 90/90 [6:15:05<00:00, 250.06s/it]



  Sanity: amp_std var=11600.3896  ✓
  ✓ Background RF activities: 1500 segments
  ✓ AR drone: 1500 segments
  ✓ Phantom drone: 1500 segments
✓ Saved 4,500 rows → dronerf_features_v5.csv

  Classes: 3
    [0] Background RF activities  (1500 samples)
    [1] AR drone  (1500 samples)
    [2] Phantom drone  (1500 samples)

FEATURE VALIDATION + SELECTION (v5: all 52)
  After RobustScaler: dropped=2  kept=50
  PCA variance:
    Top- 3 PCs: 1.000  [✓ GOOD]
    Top- 5 PCs: 1.000  [✓ GOOD]
    Top-10 PCs: 1.000  [✓ GOOD]

  Top-15 features by MI:
     1. spec_flatness                   0.3440  ★★
     2. spectral_rolloff_85             0.3419  ★★
     3. spectral_centroid               0.3364  ★★
     4. spectral_entropy                0.3258  ★★
     5. snr_like_db                     0.3166  ★★
     6. ifreq_mean                      0.3151  ★★
     7. energy_band3                    0.3120  ★★
     8. energy_band4                    0.3109  ★★
     9. spectral_spread                 0.3104 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


CONTINUAL RF DRONE MONITORING v5 — FINAL SUMMARY

7 FIXES APPLIED:
  FIX-1  window 1024→8192 samples (0.10ms→0.82ms) — captures full signal cycles
  FIX-2  skip segment-0 zero-pad transient (SDR ramp-up) in training AND inference
  FIX-3  confidence gate 0.65→0.50 — borderline drones enter trust/threat path
  FIX-4  synthetic profiles in raw feature space (computed from actual CSV medians)
  FIX-5  auto-classify unknown emitters on promotion via RF+LR ensemble
  FIX-6  dynamic trusted DB — entries upgrade SAFE_UNKNOWN → AUTO_AR_DRONE etc.
  FIX-7  all 52 features used (no MI cutoff) — keeps spectral shape features

DATASET   : 4,500 segments  |  window=8192  step=4096  |  3 classes
MODELS
  RF       : acc=0.7622  F1=0.7575  OOB=0.7369
  LR       : acc=0.6478  F1=0.6269  (0.03ms inference — replaces GNB)
  DeepRFNet: acc=0.3778  F1=0.2549
EVALUATION
  Known-drone accuracy : 93.45%
  False alarm rate     : 0.33%
  Background recall    : 73.33%
TRUSTED DB
  (empty)

System ready.
